# Can a Kaggle container join a NetBird overlay?

Decides whether the teacher can be reached over a private WireGuard overlay
instead of a public cloudflared tunnel. NetBird has no userspace netstack mode, so
if `/dev/net/tun` or `CAP_NET_ADMIN` is missing, the answer is no and the fallback
is Tailscale in userspace mode.

This kernel downloads nothing and connects to nothing: it reports capabilities.


In [ ]:
import shutil, subprocess
from pathlib import Path

def sh(cmd):
    return subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True).stdout.strip()

print("--- tun device ---")
print(sh("ls -l /dev/net/tun 2>&1 || echo MISSING"))
print("--- capability set ---")
print(sh("capsh --print 2>/dev/null | grep -i 'net_admin\|cap_net' || cat /proc/self/status | grep CapEff"))
print("--- can we create a tun interface? ---")
print(sh("ip tuntap add dev nbprobe mode tun 2>&1 && echo CREATED && ip link del nbprobe && echo REMOVED || echo REFUSED"))
print("--- am I root? ---")
print(sh("id -u"))
print("--- is netbird installable (reachable, not run) ---")
print(sh("curl -sS -o /dev/null -w '%{http_code}' --max-time 10 https://pkgs.netbird.io/install.sh 2>&1"))

tun = Path("/dev/net/tun").exists()
cap = "cap_net_admin" in sh("capsh --print 2>/dev/null").lower()
print()
print("VERDICT: tun_device =", tun, "| cap_net_admin =", cap)
print("netbird_viable =", tun and cap)
if not (tun and cap):
    print("fallback: Tailscale userspace mode (no TUN, no CAP_NET_ADMIN needed)")
